# 008: Transformer Architecture — Multi-head attention, positional encoding, and encoder block

## MULTI-HEAD ATTENTION
$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

## POSITIONAL ENCODING (Vaswani et al., 2017)
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$
---


In [ ]:
import numpy as np

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def positional_encoding(seq_len, d_model):
    """Sinusoidal positional encoding from 'Attention Is All You Need'."""
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            denom = 10000 ** (i / d_model)
            PE[pos, i] = np.sin(pos / denom)
            if i + 1 < d_model:
                PE[pos, i+1] = np.cos(pos / denom)
    return PE

class MultiHeadAttention:
    """Multi-head scaled dot-product attention from scratch."""
    def __init__(self, d_model, n_heads):
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = np.random.randn(d_model, d_model) * 0.01
        self.W_k = np.random.randn(d_model, d_model) * 0.01
        self.W_v = np.random.randn(d_model, d_model) * 0.01
        self.W_o = np.random.randn(d_model, d_model) * 0.01

    def forward(self, X):
        Q = X @ self.W_q
        K = X @ self.W_k
        V = X @ self.W_v
        seq_len = X.shape[0]
        heads = []
        for h in range(self.n_heads):
            s = h * self.d_k
            e = s + self.d_k
            scores = Q[:, s:e] @ K[:, s:e].T / np.sqrt(self.d_k)
            weights = softmax(scores, axis=-1)
            head_out = weights @ V[:, s:e]
            heads.append(head_out)
        concat = np.hstack(heads)
        return concat @ self.W_o



In [ ]:
print("--- Positional Encoding ---")
PE = positional_encoding(8, 16)
print(f"Shape: {PE.shape}")
print(f"Position 0: {PE[0, :8].round(3)}")
print(f"Position 7: {PE[7, :8].round(3)}")



In [ ]:
print("\n--- Multi-Head Attention (4 heads) ---")
np.random.seed(42)
d_model, n_heads = 16, 4
X = np.random.randn(6, d_model)  # 6 tokens
X = X + positional_encoding(6, d_model)  # Add positional info

mha = MultiHeadAttention(d_model, n_heads)
output = mha.forward(X)
print(f"Input shape:  {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Output[0]:    {output[0, :8].round(4)}")
